# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/rachanabanik/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /Users/rachanabanik/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Use-Case Data!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [8]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [9]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [10]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [11]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [12]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '81ed36'. Skipping!
Property 'summary' already exists in node '44c70e'. Skipping!
Property 'summary' already exists in node 'cfa467'. Skipping!
Property 'summary' already exists in node '192e65'. Skipping!
Property 'summary' already exists in node 'fbe86f'. Skipping!
Property 'summary' already exists in node '26a893'. Skipping!
Property 'summary' already exists in node '50c719'. Skipping!
Property 'summary' already exists in node '5580e5'. Skipping!
Property 'summary' already exists in node '3837d6'. Skipping!
Property 'summary' already exists in node 'df7a2a'. Skipping!
Property 'summary' already exists in node 'cc0191'. Skipping!
Property 'summary' already exists in node '396134'. Skipping!
Property 'summary' already exists in node 'b00149'. Skipping!
Property 'summary' already exists in node '864f0b'. Skipping!
Property 'summary' already exists in node '8ce785'. Skipping!
Property 'summary' already exists in node '461c47'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'fbe86f'. Skipping!
Property 'summary_embedding' already exists in node '192e65'. Skipping!
Property 'summary_embedding' already exists in node '26a893'. Skipping!
Property 'summary_embedding' already exists in node 'df7a2a'. Skipping!
Property 'summary_embedding' already exists in node '3837d6'. Skipping!
Property 'summary_embedding' already exists in node '81ed36'. Skipping!
Property 'summary_embedding' already exists in node 'cfa467'. Skipping!
Property 'summary_embedding' already exists in node 'cc0191'. Skipping!
Property 'summary_embedding' already exists in node '50c719'. Skipping!
Property 'summary_embedding' already exists in node '5580e5'. Skipping!
Property 'summary_embedding' already exists in node '44c70e'. Skipping!
Property 'summary_embedding' already exists in node '396134'. Skipping!
Property 'summary_embedding' already exists in node 'b00149'. Skipping!
Property 'summary_embedding' already exists in node '461c47'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 86, relationships: 711)

We can save and load our knowledge graphs as follows.

In [13]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 86, relationships: 711)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [14]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [15]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.

1. SingleHopSpecificQuerySynthesizer (50% of queries)
What it does: Creates questions that can be answered by looking at just one piece of information from the documents.
Simple explanation: These are straightforward questions that ask for specific facts or details that are directly mentioned in the text. You don't need to connect multiple pieces of information to answer them.
Example from the notebook: "What does the research by Bick et al. (2024) reveal about ChatGPT usage patterns?"
2. MultiHopAbstractQuerySynthesizer (25% of queries)
What it does: Creates questions that require connecting multiple pieces of information, but the questions are more general or conceptual in nature.
Simple explanation: These questions ask you to think about broader patterns, trends, or concepts by combining information from different parts of the documents.
Example from the notebook: "How do the large language models (LLMs) like ChatGPT impact work productivity?"
3. MultiHopSpecificQuerySynthesizer (25% of queries)
What it does: Creates questions that require connecting multiple specific pieces of information to answer.
Simple explanation: These questions ask for specific details, but you need to look at multiple parts of the documents and connect the information together to get the complete answer.
Example from the notebook: "How many messages sent to ChatGPT were related to work activities in June 2024?"
The key difference is:
Single Hop = One piece of information needed
Multi Hop = Multiple pieces of information needed
Specific = Looking for concrete facts and details
Abstract = Looking for broader concepts and patterns


Finally, we can use our `TestSetGenerator` to generate our testset!

In [16]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is the significance of July 2025 in the c...,[Introduction ChatGPT launched in November 202...,"By July 2025, 18 billion messages were being s...",single_hop_specifc_query_synthesizer
1,What is the significance of June 2025 in the c...,[Table 1: ChatGPT daily message counts (millio...,The report provides 7-day average daily messag...,single_hop_specifc_query_synthesizer
2,What does SOC2 code 15 refer to in the context...,[Variation by Occupation Figure 23 presents va...,SOC2 code 15 refers to computer-related occupa...,single_hop_specifc_query_synthesizer
3,How does the term 'Doing' relate to user inter...,[Conclusion This paper studies the rapid growt...,"In the context of ChatGPT usage, the term 'Doi...",single_hop_specifc_query_synthesizer
4,Based on the data showing ChatGPT's increasing...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,The data indicates that nearly 10% of ChatGPT ...,multi_hop_abstract_query_synthesizer
5,H0w do usage patterns and growht across popuLa...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The context indicates that ChatGPT's rapid ado...,multi_hop_abstract_query_synthesizer
6,Based on the increasing percentage of non-work...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The data shows that non-work messages have gro...,multi_hop_abstract_query_synthesizer
7,"Whay is Handa et al., 2025, so important for u...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,"Handa et al., 2025, is important because it pr...",multi_hop_specific_query_synthesizer
8,"Whay is Handa et al., 2025 report about ChatGP...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,"Handa et al., 2025 report that ChatGPT has rap...",multi_hop_specific_query_synthesizer
9,How has the adoption and usage of ChatGPT by J...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT had been used weekly by ...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [17]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '6e1c9b'. Skipping!
Property 'summary' already exists in node 'fe9f4a'. Skipping!
Property 'summary' already exists in node 'ca259d'. Skipping!
Property 'summary' already exists in node '591501'. Skipping!
Property 'summary' already exists in node '284772'. Skipping!
Property 'summary' already exists in node '1b09e5'. Skipping!
Property 'summary' already exists in node 'a26485'. Skipping!
Property 'summary' already exists in node '7e58c5'. Skipping!
Property 'summary' already exists in node '712d97'. Skipping!
Property 'summary' already exists in node '81c648'. Skipping!
Property 'summary' already exists in node '9f3234'. Skipping!
Property 'summary' already exists in node '479094'. Skipping!
Property 'summary' already exists in node 'ca7d17'. Skipping!
Property 'summary' already exists in node '318201'. Skipping!
Property 'summary' already exists in node 'b39c01'. Skipping!
Property 'summary' already exists in node '9b9351'. Skipping!
Property

...

Property 'summary_embedding' already exists in node '7e58c5'. Skipping!
Property 'summary_embedding' already exists in node 'a26485'. Skipping!
Property 'summary_embedding' already exists in node 'fe9f4a'. Skipping!
Property 'summary_embedding' already exists in node '479094'. Skipping!
Property 'summary_embedding' already exists in node '284772'. Skipping!
Property 'summary_embedding' already exists in node '6e1c9b'. Skipping!
Property 'summary_embedding' already exists in node 'ca7d17'. Skipping!
Property 'summary_embedding' already exists in node '591501'. Skipping!
Property 'summary_embedding' already exists in node 'ca259d'. Skipping!
Property 'summary_embedding' already exists in node '9f3234'. Skipping!
Property 'summary_embedding' already exists in node '1b09e5'. Skipping!
Property 'summary_embedding' already exists in node '712d97'. Skipping!
Property 'summary_embedding' already exists in node 'b39c01'. Skipping!
Property 'summary_embedding' already exists in node '318201'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [18]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,"Bick et al., 2024 what they say about ChatGPT ...",[Introduction ChatGPT launched in November 202...,"The context states that Bick et al., 2024, is ...",single_hop_specifc_query_synthesizer
1,How does the data on ChatGPT message usage in ...,[Table 1: ChatGPT daily message counts (millio...,The data indicates that while messages have gr...,single_hop_specifc_query_synthesizer
2,Section what mean in ChatGPT use?,[Variation by Occupation Figure 23 presents va...,The context discusses variation in ChatGPT usa...,single_hop_specifc_query_synthesizer
3,How does the concept of Personal Reflection re...,[Conclusion This paper studies the rapid growt...,The context indicates that the study of ChatGP...,single_hop_specifc_query_synthesizer
4,How do Large Language Models (LLMs) like ChatG...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"ChatGPT, based on a Large Language Model (LLM)...",multi_hop_abstract_query_synthesizer
5,"How does the increase in non-work messages, wh...",[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The data indicates that non-work messages have...,multi_hop_abstract_query_synthesizer
6,Considering the rapid adoption of ChatGPT and ...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"The rapid adoption of ChatGPT, with over 18 bi...",multi_hop_abstract_query_synthesizer
7,how classify message types ChatGPT use for wor...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,ChatGPT messages are classified into categorie...,multi_hop_abstract_query_synthesizer
8,"Based on Handa et al. (2025), how does the gro...",[<1-hop>\n\nTable 1: ChatGPT daily message cou...,Handa et al. (2025) report that total ChatGPT ...,multi_hop_specific_query_synthesizer
9,How does Handa et al. (2025) describe ChatGPT'...,[<1-hop>\n\nTable 1: ChatGPT daily message cou...,Handa et al. (2025) report that ChatGPT's tota...,multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [19]:
from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE8"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [20]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [21]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [22]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [23]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [24]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [25]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [26]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [27]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [28]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [29]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

'Based on the provided context, people are using AI, particularly generative AI like ChatGPT, in a variety of ways both at work and outside of work. At work, AI is used to perform workplace tasks by either augmenting or automating human labor, improving productivity, producing writing, software code, spreadsheets, and other digital products. Users interact with AI for different intents such as asking questions, getting tasks done, or expressing themselves. The flexibility of generative AI distinguishes it from traditional technologies and web search engines. Additionally, AI serves roles as co-workers producing output and as co-pilots giving advice and helping with problem-solving. Outside of work, uses include self-expression activities such as relationships, personal reflection, gaming, and role play. Therapy and companionship are also noted as prevalent use cases for generative AI.\n\nIn summary, people are using AI to augment workplace productivity, automate tasks, produce various 

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [30]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [31]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`: Evaluates: Basic question-answering accuracy
Uses LangSmith's built-in "qa" evaluator
Compares the generated answer against the reference answer
Determines if the response correctly answers the question
This is a standard accuracy metric for RAG systems
- `labeled_helpfulness_evaluator`: Evaluates: How helpful the response is to the user
Uses a custom criteria called "helpfulness"
Asks: "Is this submission helpful to the user, taking into account the correct reference answer?"
Considers both correctness AND usefulness to the user
Takes into account the reference answer to determine if the response is genuinely helpful
- `dopeness_evaluator`: Evaluates: How engaging and non-generic the response is
Uses a custom criteria called "dopeness"
Asks: "Is this response dope, lit, cool, or is it just a generic response?"
Measures the quality and creativity of the response
Distinguishes between engaging, interesting responses vs. bland, generic ones
This is a fun way to measure response quality beyond just accuracy

## LangSmith Evaluation

In [32]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'earnest-match-2' at:
https://smith.langchain.com/o/77ce7a24-10c1-4621-b352-72f530436ef8/datasets/11533d4c-0b26-4038-ab22-0e018bc60f59/compare?selectedSessions=8e7d7336-1224-4514-ae9b-5b1ff901bb81




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,Based on the data showing that over 70% of Cha...,"Based on the provided context, the fact that o...",None,"The data indicates that in the US, more than 7...",1,1,0,4.983586,1919c00d-95b7-4e65-a666-afa968562fc1,79a1aa70-710c-4b06-b431-d92eda27337e
1,How does the growth of ChatGPT's usage in the ...,Based on the context provided:\n\nThe growth o...,None,"The context indicates that as of July 2025, ab...",1,1,0,6.062347,329b8527-60fe-4723-ab5e-f8994ab0d284,04384b38-372d-47b1-ac0b-08482943ae7a
2,How does Handa et al. (2025) describe ChatGPT'...,I don't know.,None,Handa et al. (2025) report that ChatGPT's tota...,0,0,0,0.827516,24043bdc-37df-45f8-9b1a-d3ce339d3489,61405192-d90b-4ddf-9afd-9bb533a5433f
3,"Based on Handa et al. (2025), how does the gro...","Based on the provided context, the growth in C...",None,Handa et al. (2025) report that total ChatGPT ...,1,1,0,5.893266,a44ecc25-5fdc-4dac-b21c-d95c1552a257,358702e4-00d8-4550-9efd-51c80b4401c8
4,how classify message types ChatGPT use for wor...,ChatGPT message types used for work are classi...,None,ChatGPT messages are classified into categorie...,1,1,0,4.457889,d4ede933-c7a6-4477-bbb2-919340fb4df7,bc3033c7-42bb-4c39-9534-f094afb3c736
5,Considering the rapid adoption of ChatGPT and ...,"Based on the provided context, the rapid adopt...",None,"The rapid adoption of ChatGPT, with over 18 bi...",1,0,0,8.361115,0aa2a09f-4a01-4611-8c2d-472e3dcff2ae,e7b51ed1-d8dc-4939-84e6-be6c7b12a938
6,"How does the increase in non-work messages, wh...","The increase in non-work messages, which have ...",None,The data indicates that non-work messages have...,1,1,0,3.786935,0827b48d-d139-4cc3-ad8a-bc8f50dc6fd2,767bd04d-1914-4586-aef6-68b7b404fc08
7,How do Large Language Models (LLMs) like ChatG...,Large Language Models (LLMs) like ChatGPT exem...,None,"ChatGPT, based on a Large Language Model (LLM)...",1,1,0,6.553039,b3d3f030-7bc5-4c5d-932a-da1b6a6872e7,f852b250-7a1d-4b9a-8c72-3374e48e683b
8,How does the concept of Personal Reflection re...,The concept of Personal Reflection appears as ...,None,The context indicates that the study of ChatGP...,1,1,0,2.904031,c179885e-f747-4931-a534-6e76f42467b1,5ddbe144-9f2c-4e2a-866c-f61d46cfc5e4
9,Section what mean in ChatGPT use?,I don't know,None,The context discusses variation in ChatGPT usa...,0,0,0,0.617270,83cac55c-1063-4ae6-90ab-bd032184450c,dbf0fba9-3b96-49b4-b6b0-228ff6a78472


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [33]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [34]:
rag_documents = docs

In [35]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

1. Information Completeness
Smaller chunks (500 chars): May contain incomplete information, leading to fragmented answers
Larger chunks (1000 chars): More likely to contain complete context and relationships between concepts
Impact: Larger chunks help the LLM understand the full context of a topic
2. Retrieval Quality
Smaller chunks: Higher precision but may miss related information that's split across chunks
Larger chunks: Better recall as related information stays together, but may include irrelevant details
Impact: The retriever can find more comprehensive context with larger chunks
3. Answer Quality
Smaller chunks: May produce incomplete or fragmented answers
Larger chunks: Enable more coherent, complete responses with better context
Impact: The LLM has more context to generate comprehensive answers
4. Trade-offs
Computational cost: Larger chunks mean more tokens processed per retrieval
Relevance: Larger chunks might include less relevant information
Memory usage: Higher memory consumption with larger chunks
5. Specific to This Use Case
In the notebook's example:
Original: 500 chars → might split important relationships
Modified: 1000 chars → keeps related concepts together
Result: Better performance on complex questions that require understanding relationships between concepts
Bottom Line: Larger chunks generally improve answer quality by providing more complete context, but at the cost of computational efficiency and potential noise from irrelevant information.

In [36]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

1. Embedding Quality & Dimensionality
Small model: Lower-dimensional embeddings (typically 512-1536 dimensions)
Large model: Higher-dimensional embeddings (typically 3072 dimensions)
Impact: Larger models capture more nuanced semantic relationships and subtle meaning differences
2. Semantic Understanding
Small model: Basic semantic similarity, may miss subtle context differences
Large model: Better understanding of complex relationships, synonyms, and contextual nuances
Impact: More accurate retrieval of semantically similar content
3. Retrieval Precision
Small model: May retrieve somewhat relevant but not optimal chunks
Large model: Better at distinguishing between closely related concepts
Impact: Higher precision in finding the most relevant information
4. Complex Query Handling
Small model: Struggles with complex, multi-faceted queries
Large model: Better at understanding sophisticated question patterns
Impact: Improved performance on complex questions that require deep semantic understanding
5. Trade-offs
Computational cost: Large models are more expensive to run
Latency: Slower embedding generation and similarity search
Storage: Higher storage requirements for vector databases
Memory: More memory needed for processing
6. Specific Performance Improvements
The large model typically provides:
Better semantic clustering of related concepts
Improved handling of synonyms and paraphrases
More accurate ranking of retrieved documents
Better performance on domain-specific terminology
7. Real-world Impact
Retrieval accuracy: Finds more relevant chunks for complex questions
Answer quality: Better context leads to more accurate and complete answers
User satisfaction: More relevant results improve overall system performance
Bottom Line: The larger embedding model provides significantly better semantic understanding and retrieval accuracy, leading to higher-quality answers, but at the cost of increased computational resources and latency.




In [37]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [38]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [39]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [40]:
dopeness_rag_chain.invoke({"question" : "How are people using AI to make money?"})

'Alright, here’s the ultra-rad scoop on how folks are cashing in with AI, straight from the digital trenches of this next-level dossier:\n\nPeople are not just using AI to grind through tasks—nah, they’re leveling up their hustle by treating ChatGPT like a slick advisor or a research wizard sidekick. The magic sauce? AI boosts worker output by sharpening decision-making, especially in knowledge-heavy gigs where boss-level choices translate directly into fat stacks of productivity and, ultimately, $$$.\n\nSo instead of just automating menial chores, AI flexes as a decision support system that turbocharges your brainpower. This means smarter moves, faster research, and clearer strategies—transforming your work game from "meh" to "money moves." The numbers back it up: the US consumer surplus from using generative AI hits a staggering $97 billion a year. That’s trillions in value unlocked by folks using AI to augment their skills and get paid more for smarter, not harder, work.\n\nIn short

Finally, we can evaluate the new chain on the same test set!

In [41]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'excellent-reward-57' at:
https://smith.langchain.com/o/77ce7a24-10c1-4621-b352-72f530436ef8/datasets/11533d4c-0b26-4038-ab22-0e018bc60f59/compare?selectedSessions=fef562da-2ed0-4da3-89d8-d8206dd12b63




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,Based on the data showing that over 70% of Cha...,"Yo, here’s the 411 straight from the juice: Ov...",None,"The data indicates that in the US, more than 7...",1,1,1,7.640575,1919c00d-95b7-4e65-a666-afa968562fc1,d8de4f55-3ef1-4464-b662-be294c711988
1,How does the growth of ChatGPT's usage in the ...,"Alright, buckle up because the story of ChatGP...",None,"The context indicates that as of July 2025, ab...",1,1,1,6.097258,329b8527-60fe-4723-ab5e-f8994ab0d284,34895737-4717-4054-9deb-e7c137776669
2,How does Handa et al. (2025) describe ChatGPT'...,"Yo, here’s the ultimate scoop from Handa et al...",None,Handa et al. (2025) report that ChatGPT's tota...,1,0,1,4.704521,24043bdc-37df-45f8-9b1a-d3ce339d3489,a18091db-d105-4e48-a7fe-8b186a719a32
3,"Based on Handa et al. (2025), how does the gro...","Yo, let’s break down the cosmic growth story f...",None,Handa et al. (2025) report that total ChatGPT ...,1,1,1,9.792506,a44ecc25-5fdc-4dac-b21c-d95c1552a257,60a941cd-6761-4d22-8c0b-cbc8dc84e064
4,how classify message types ChatGPT use for wor...,"Yo, let’s crank this up to next-level dopeness...",None,ChatGPT messages are classified into categorie...,1,1,1,7.426780,d4ede933-c7a6-4477-bbb2-919340fb4df7,1ce2cc3d-303c-4f94-8fb1-c6c2800e7a5b
5,Considering the rapid adoption of ChatGPT and ...,"Alright, let’s slice through the noise and dro...",None,"The rapid adoption of ChatGPT, with over 18 bi...",1,1,1,6.795936,0aa2a09f-4a01-4611-8c2d-472e3dcff2ae,e208d483-519e-4084-93c8-607326990faf
6,"How does the increase in non-work messages, wh...","Oh snap, let’s break down this rad surge in Ch...",None,The data indicates that non-work messages have...,1,1,1,4.050904,0827b48d-d139-4cc3-ad8a-bc8f50dc6fd2,19880962-ba92-4e56-a6e6-d971d5ab7fe4
7,How do Large Language Models (LLMs) like ChatG...,"Alright, buckle up for some AI knowledge dropp...",None,"ChatGPT, based on a Large Language Model (LLM)...",1,1,1,6.454268,b3d3f030-7bc5-4c5d-932a-da1b6a6872e7,c1e989f0-1799-4b73-8dbb-280f684f37bf
8,How does the concept of Personal Reflection re...,"Yo, strap in for this crisp insight on **Perso...",None,The context indicates that the study of ChatGP...,1,1,1,5.745508,c179885e-f747-4931-a534-6e76f42467b1,12527103-4ab4-43a6-a913-601bf81356ce
9,Section what mean in ChatGPT use?,"Yo, here’s the dealio breaking down what “Sect...",None,The context discusses variation in ChatGPT usa...,1,0,1,6.081984,83cac55c-1063-4ae6-90ab-bd032184450c,37a52563-52af-479e-8c0a-1e67ed40677e


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.


Detailed Analysis:
1. Helpfulness Evaluator
Result: Both chains scored "Y" (Yes) for helpfulness
Explanation: The improved chain maintained the same level of helpfulness while adding the "dope" factor
Why: Larger chunks and better embeddings provided more complete context, maintaining helpfulness
2. Correctness Evaluator (QA)
Result: Both chains scored "CORRECT" for accuracy
Explanation: The technical improvements (larger chunks, better embeddings) maintained accuracy
Why: Better retrieval and context didn't compromise factual correctness
3. Dopeness Evaluator
Result: Original: "N" (No) → Improved: "Y" (Yes)
Explanation: MASSIVE IMPROVEMENT - This was the biggest success!
Why: The "dope" prompt directly optimized for this metric, encouraging engaging, non-generic responses
Sample Response Comparison:
Original Chain Response:
> "The increase in non-work messages, which have grown from 53% in June 2024 to over 70% (specifically 73% by June 2025)..."
Improved Chain Response:
> "Oh snap, let's break down this rad surge in ChatGPT action! Between June 2024 and June 2025, overall message volume exploded—from a chill 451 million daily messages to a wild 2,627 million..."
Why These Changes Worked:
Larger Chunks (500→1000 chars): Provided more complete context without losing accuracy
Better Embeddings (small→large): Improved retrieval quality while maintaining correctness
"Dope" Prompt: Directly targeted the engagement metric, transforming generic responses into engaging ones
Key Takeaway:
The improvements successfully enhanced user engagement (dopeness) while maintaining both accuracy and helpfulness. This demonstrates that technical optimizations can be combined with prompt engineering to achieve multiple quality improvements simultaneously.




## Screenshots of LangSmith Evaluation Results

Here are the screenshots showing the comparison between the two RAG chains from the LangSmith evaluation:

### LangSmith Evaluation Comparison
![LangSmith Evaluation Results](Screenshot 2025-10-06 at 1.59.50 PM.png)

![LangSmith Evaluation Results](Screenshot 2025-10-06 at 2.00.07 PM.png)

**Note:** In these LangSmith comparison screenshots:
- **Left column**: Improved chain (with dopeness prompt, larger chunks, and text-embedding-3-large)
- **Right column**: Original chain (baseline without dopeness)

These screenshots show the LangSmith evaluation results comparing the baseline RAG chain and the improved "dopeness" chain, demonstrating how the modifications to chunk size, embedding model, and prompt engineering impacted the evaluation metrics in the LangSmith dashboard.
